# YOLO26 Hand-Gesture Detection: HaGRID 9-Class Training

Trains a [YOLO26](https://docs.ultralytics.com/models/yolo26/) object detector to recognize nine static hand
gestures from the HaGRID dataset.

| | |
|---|---|
| **Classes** | call, dislike, fist, like, ok, palm, peace, rock, stop |
| **Images** | 15,786 total: 12,835 train / 1,388 val / 1,563 test |
| **Split** | By participant (SHA-256 hash of seed + participant ID, seed 42), so no person appears in more than one split |
| **Label format** | [YOLO detection format](https://docs.ultralytics.com/datasets/detect/) (`class cx cy w h`, normalized 0-1) |
| **Runtime** | Google Colab with a GPU (Runtime > Change runtime type) |

**Before running:** upload the dataset zip to Google Drive and set `DATASET_ZIP` in Section 2.

**Contents**
1. Setup
2. Configuration
3. Load the dataset
4. Verify the dataset
5. Visual check
6. Smoke test
7. Full training
8. Evaluation
9. Save the model

## 1. Setup

Installs the Ultralytics package, which provides YOLO26, and confirms the GPU is visible.

In [ ]:
!pip install -q -U ultralytics
import ultralytics
from ultralytics import YOLO

ultralytics.checks()

## 2. Configuration

Every setting you might change lives in this one cell, so the rest of the notebook can run unmodified.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# --- Edit these paths ---
DATASET_ZIP = Path("/content/drive/MyDrive/hgr-game-control/hagrid9_yolo.zip")
SAVE = Path("/content/drive/MyDrive/hgr-game-control")

# --- Locations ---
EXTRACT_DIR = Path("/content/dataset")
RUNS_DIR = SAVE / "runs"

# --- Model and training settings ---
MODEL_WEIGHTS = "yolo26n.pt"
RUN_NAME = "gestures9_n"
IMG_SIZE = 640
EPOCHS = 100
PATIENCE = 10
BATCH = 32
RESUME = False

SAVE.mkdir(parents=True, exist_ok=True)
assert DATASET_ZIP.exists(), f"Dataset zip not found: {DATASET_ZIP}"

## 3. Load the dataset

Unzips the dataset and points its `data.yaml` at the Colab file locations.

The `data.yaml` that came with the dataset still holds the path from the machine it was created on
(`D:/hgr-work/...`), which does not exist in Colab. Instead of editing the original, this cell writes a corrected
copy named `data_colab.yaml` and trains from that.

Optimized for MacOS systems.

In [ ]:
import yaml

!unzip -q -o "{DATASET_ZIP}" -d "{EXTRACT_DIR}"

src_yaml = next(p for p in EXTRACT_DIR.rglob("data.yaml") if "__MACOSX" not in p.parts)
DATA_ROOT = src_yaml.parent
print("Dataset root:", DATA_ROOT)

for f in DATA_ROOT.rglob("*.cache"):
    f.unlink()

cfg = yaml.safe_load(src_yaml.read_text())
CLASSES = cfg["names"]
if isinstance(CLASSES, dict):
    CLASSES = [CLASSES[k] for k in sorted(CLASSES)]

colab_cfg = {
    "path": str(DATA_ROOT),
    "train": cfg["train"],
    "val": cfg["val"],
    "test": cfg["test"],
    "names": {i: name for i, name in enumerate(CLASSES)},
}
DATA_YAML = DATA_ROOT / "data_colab.yaml"
DATA_YAML.write_text(yaml.safe_dump(colab_cfg, sort_keys=False))
print(DATA_YAML.read_text())

## 4. Verify the dataset

Checks the extracted data against `summary.json`, which was written when the dataset was built. This confirms the
zip is complete and that every class ID in the label files means what we think it means.

For each split (train / val / test), the cell checks that:
- every image has a matching label file (a label file can be empty, which YOLO treats as "background only"), and
- the number of boxes per class in the label files equals the count recorded in `summary.json`.

In [ ]:
import json
from collections import Counter

IMG_EXTS = {".jpg", ".jpeg", ".png"}
summary = json.loads((DATA_ROOT / "summary.json").read_text())
assert summary["classes"] == CLASSES, "Class order in summary.json does not match data.yaml"

IMAGES, LABELS = {}, {}
for split in ["train", "val", "test"]:
    IMAGES[split] = {p.stem: p for p in (DATA_ROOT / "images" / split).rglob("*") if p.suffix.lower() in IMG_EXTS}
    LABELS[split] = {p.stem: p for p in (DATA_ROOT / "labels" / split).rglob("*.txt")}

    boxes = Counter()
    for label_file in LABELS[split].values():
        for line in label_file.read_text().splitlines():
            if line.strip():
                boxes[CLASSES[int(line.split()[0])]] += 1

    expected = summary["target_boxes_by_class"][split]
    mismatches = [c for c in CLASSES if boxes[c] != expected[c]]
    no_label = set(IMAGES[split]) - set(LABELS[split])

    print(f"{split:5s} | images: {len(IMAGES[split]):6d} | label files: {len(LABELS[split]):6d} | "
          f"images without label: {len(no_label)} | class count mismatches: {mismatches or 'none'}")

print("Expected total images:", summary["total_images"])

## 5. Visual check

Draws the label boxes on random training images. This catches problems that counts cannot, such as boxes shifted
off the hand or the wrong gesture name.

In [ ]:
import random
import cv2
import matplotlib.pyplot as plt


def show_samples(split="train", n=6, seed=None):
    rng = random.Random(seed)
    stems = rng.sample(sorted(IMAGES[split]), n)
    fig, axes = plt.subplots(2, 3, figsize=(12, 6))
    for ax, stem in zip(axes.ravel(), stems):
        im = cv2.cvtColor(cv2.imread(str(IMAGES[split][stem])), cv2.COLOR_BGR2RGB)
        H, W = im.shape[:2]
        label_file = LABELS[split].get(stem)
        lines = label_file.read_text().splitlines() if label_file else []
        for line in filter(str.strip, lines):
            c, cx, cy, w, h = line.split()
            cx, cy, w, h = float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
            p1 = (int(cx - w / 2), int(cy - h / 2))
            p2 = (int(cx + w / 2), int(cy + h / 2))
            cv2.rectangle(im, p1, p2, (0, 255, 0), 2)
            cv2.putText(im, CLASSES[int(c)], (p1[0], max(p1[1] - 5, 12)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        ax.imshow(im)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


show_samples("train")

## 6. Smoke test

A very short training run to confirm the whole pipeline works (dataset loads, GPU is used, results save to Drive)
before committing to the full run.

In [ ]:
model = YOLO(MODEL_WEIGHTS)
model.train(
    data=str(DATA_YAML),
    epochs=2,
    fraction=0.05,
    imgsz=IMG_SIZE,
    batch=16,
    workers=2,
    project=str(RUNS_DIR),
    name="smoke_test",
)

## 7. Full training

The real run, using the settings from Section 2.


**If Colab disconnects mid-run:** reconnect, re-run Sections 1-3 (Sections 4-6 can be skipped), set `RESUME = True` in Section 2, and run this
cell again. It reloads `last.pt` and continues from the epoch where it stopped. If Ultralytics had to rename the
run folder (for example `gestures9_n2`), set `RUN_NAME` to match.

In [ ]:
if RESUME:
    model = YOLO(RUNS_DIR / RUN_NAME / "weights" / "last.pt")
    results = model.train(resume=True)
else:
    model = YOLO(MODEL_WEIGHTS)
    results = model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        patience=PATIENCE,
        imgsz=IMG_SIZE,
        batch=BATCH,
        workers=2,
        project=str(RUNS_DIR),
        name=RUN_NAME,
    )

## 8. Evaluation

Shows the training curves, then scores the best checkpoint on both the validation and the held-out test split.

In [ ]:
import pandas as pd
from IPython.display import Image as IPImage, display

# List the runs saved in Drive, then point run_dir at the training run
print(sorted(p.name for p in RUNS_DIR.iterdir()))
run_dir = RUNS_DIR / RUN_NAME   # change if the list shows e.g. "gestures9_n2"
assert (run_dir / "weights" / "best.pt").exists(), f"No best.pt in {run_dir}"

display(IPImage(filename=str(run_dir / "results.png")))
display(IPImage(filename=str(run_dir / "confusion_matrix_normalized.png")))

best = YOLO(run_dir / "weights" / "best.pt")

for split in ["val", "test"]:
    metrics = best.val(data=str(DATA_YAML), split=split, project=str(RUNS_DIR), name=f"{RUN_NAME}_{split}")
    print(f"\n{split.upper()}: mAP50 = {metrics.box.map50:.3f} | mAP50-95 = {metrics.box.map:.3f}")

per_class = pd.DataFrame({
    "class": [metrics.names[i] for i in metrics.box.ap_class_index],
    "mAP50": metrics.box.ap50,
    "mAP50-95": metrics.box.ap,
}).round(3)
print("\nPer-class results (test split):")
display(per_class)

## 9. Save the model

Copies the best weights to a clearly named file in Drive, ready to download for local webcam inference or to upload
to GitHub.

In [ ]:
import shutil

MODELS_DIR = SAVE / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

final_path = MODELS_DIR / f"{Path(MODEL_WEIGHTS).stem}_hagrid9_best.pt"
shutil.copy(run_dir / "weights" / "best.pt", final_path)
print("Saved:", final_path)